# Flow past a cylinder: drag and vortex shedding

*fluids · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/fluids/cylinder_drag)

A lattice Boltzmann (D2Q9) simulation of flow past a cylinder. Populations $f_{xyi}$ collide towards an einsum-built equilibrium and stream with torch.roll; the force on the cylinder comes from the momentum handed over by every population that bounces off it. Above Re ≈ 47 the wake becomes a Kármán vortex street: the drag pulses and the lift swings from side to side.

**Vector form**

$$\partial_t\mathbf u + (\mathbf u\cdot\nabla)\mathbf u = -\tfrac1\rho\nabla p + \nu\nabla^2\mathbf u,\qquad C_D = \frac{F_x}{\tfrac12\rho U^2 D},\quad C_L = \frac{F_y}{\tfrac12\rho U^2 D}$$

**Index (tensor) form**

$$f_i(\mathbf x + \mathbf e_i, t + 1) = f_i - \tfrac1\tau\big(f_i - f_i^\text{eq}\big),\quad f^\text{eq}_{xyi} = w_i\,\rho\,\big(1 + 3e_{ia}u_a + \tfrac92(e_{ia}u_a)^2 - \tfrac32u_au_a\big),\quad F_a = 2\,L_{xyi}\,f_{xyi}\,e_{ia}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $f_{xyi}$ | `f` | (Nx, Ny, 9) | populations: fluid at node $(x, y)$ moving with lattice velocity $i$ |
| $e_{ia},\ w_i$ | `lattice(dev)` | (9, 2), (9,) | D2Q9 velocities (rest, 4 axes, 4 diagonals) and weights |
| $\rho = f_{xyi}\,\mathbb 1_i$ | `einsum('xyi->xy', f)` | (Nx, Ny) | density: sum over $i$ |
| $\rho u_a = f_{xyi}e_{ia}$ | `einsum('xyi,ia->xya', f, e)` | (Nx, Ny, 2) | repeated $i$ ⇒ summed: momentum density |
| $e_{ia}u_a$ | `einsum('ia,xya->xyi', e, u)` | (Nx, Ny, 9) | velocity projected on each lattice direction |
| $f(\mathbf x + \mathbf e_i)$ | `torch.roll(f[..., i], (ex, ey))` | (Nx, Ny) | streaming: shift each population one link |
| $L_{xyi}$ | `boundary_links(solid, e)` | (Nx, Ny, 9) | 1 where a population streamed from fluid into the cylinder |
| $2\,L_{xyi}f_{xyi}e_{ia}$ | `2 * einsum('xyi,xyi,ia->a', L, f, e)` | (2,) | momentum exchange: force on the cylinder (bounce-back reverses each hit) |

Lattice units: node spacing and time step 1, cylinder diameter $D = 10$, inflow $U = 0.12$; the viscosity $\nu = UD/\mathrm{Re}$ sets the relaxation time $\tau = 3\nu + \tfrac12$. The model reproduces Navier–Stokes when $U$ is well below the lattice sound speed $1/\sqrt3$. Inflow is a fixed equilibrium, outflow copies its neighbour, top and bottom are periodic (a row of cylinders, spacing $H$), and the cylinder uses full-way bounce-back, a staircase approximation of the circle. The momentum-exchange force counts every population that hits the cylinder: it reverses, so it hands over $2 f_i\mathbf e_i$. Because the channel is narrow ($D/H \approx 0.18$) the drag is higher than the free-stream $C_D \approx 1.4$ at Re = 100, and the shedding a little faster ($St \approx 0.16$–$0.2$). The first few hundred steps are an acoustic start-up transient; read the averages from the second half. A small sideways kick at the start triggers the shedding sooner.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/fluids/cylinder_drag/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import plot, solve, tensors
from .equation import viscosity_to_tau

NX, NY, D, U_IN = 180, 56, 10, 0.12  # lattice size, cylinder diameter (nodes), inflow speed (lattice units)
CENTRE = (40.0, NY / 2)


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    re, n_steps = params["re"], params["steps"]
    nu = U_IN * D / re  # lattice viscosity from Re = U D / ν
    tau = viscosity_to_tau(nu)
    e, w, opp = tensors.lattice(dev)  # (9, 2), (9,), (9,)
    solid = tensors.cylinder(NX, NY, *CENTRE, D / 2, dev)  # (Nx, Ny) bool
    links = tensors.boundary_links(solid, e)  # (Nx, Ny, 9)
    f = tensors.initial_populations(NX, NY, U_IN, e, w, dev)  # (Nx, Ny, 9)

    result = solve.simulate(f, solid, links, e, w, opp, tau, U_IN, n_steps, save_every=max(1, n_steps // 100))
    coeffs = solve.coefficients(result["force"], U_IN, D)  # (T, 2): C_D, C_L
    late = coeffs[n_steps // 2 :]

    print(
        f"lattice {NX}x{NY}, D = {D}, U = {U_IN}, Re = {re:.0f}, nu = {nu:.4f}, tau = {tau:.3f}, "
        f"blockage D/H = {D / NY:.2f}"
    )
    print(f"second half of the run: mean C_D = {late[:, 0].mean():.2f}, mean C_L = {late[:, 1].mean():+.3f}, "
          f"C_L amplitude {late[:, 1].abs().max():.2f}")  # fmt: skip
    print(
        f"Strouhal number St = f D / U = {solve.strouhal(coeffs[:, 1], U_IN, D):.3f}  (free cylinder at Re 100: ≈ 0.16)"
    )
    return plot.animate(
        result["vorticity"], result["steps"], coeffs, CENTRE, D / 2, U_IN, out_dir / "cylinder_drag.gif"
    )


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/fluids/cylinder_drag/equation.py
r"""Physics → tensor translation.

Lattice Boltzmann (D2Q9, BGK). Instead of solving Navier–Stokes for u directly, we track
populations f_{xyi}: the amount of fluid at node (x, y) moving with lattice velocity e_{ia}
(i = 0..8, a ∈ {x, y}). The macroscopic fields are moments (sums over i):

    ρ_{xy} = Σ_i f_{xyi},        ρ u_{xya} = Σ_i f_{xyi} e_{ia}

Each step relaxes f towards a local equilibrium (collision) and moves it one link (streaming):

    f_{xyi}^{eq} = w_i ρ (1 + 3 e_{ia} u_a + 9/2 (e_{ia} u_a)^2 - 3/2 u_a u_a)
    f_i(\mathbf x + \mathbf e_i, t + 1) = f_i - (f_i - f_i^{eq}) / τ,        ν = (τ - ½) / 3

which reproduces the Navier–Stokes equations at low Mach number.

Drag by momentum exchange: every population that hits the cylinder along a boundary link bounces
back, handing the body twice its momentum: F_a = 2 Σ_{xyi} L_{xyi} f_{xyi} e_{ia}.
"""

import torch


def density(f: torch.Tensor) -> torch.Tensor:
    r"""ρ_{xy} = Σ_i f_{xyi}.   (Nx, Ny, 9) → (Nx, Ny)"""
    return torch.einsum("xyi->xy", f)


def momentum(f: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    r"""ρu_{xya} = Σ_i f_{xyi} e_{ia}  (i summed).   (Nx, Ny, 9), (9, 2) → (Nx, Ny, 2)"""
    return torch.einsum("xyi,ia->xya", f, e)


def equilibrium(rho: torch.Tensor, u: torch.Tensor, e: torch.Tensor, w: torch.Tensor) -> torch.Tensor:
    r"""f^{eq}_{xyi} = w_i ρ (1 + 3 e_{ia}u_a + 4.5 (e_{ia}u_a)^2 - 1.5 u_a u_a).   → (Nx, Ny, 9)"""
    eu = torch.einsum("ia,xya->xyi", e, u)  # e_ia u_a, a summed
    uu = torch.einsum("xya,xya->xy", u, u)  # u_a u_a
    return torch.einsum("i,xy,xyi->xyi", w, rho, 1 + 3 * eu + 4.5 * eu**2 - 1.5 * uu[..., None])


def collide(f: torch.Tensor, f_eq: torch.Tensor, tau: float) -> torch.Tensor:
    """BGK collision: relax towards equilibrium with relaxation time τ."""
    return f - (f - f_eq) / tau


def stream(f: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    """Move every population one node along its own velocity e_i (periodic wrap-around via torch.roll)."""
    return torch.stack([torch.roll(f[..., i], shifts=(int(ex), int(ey)), dims=(0, 1))
                        for i, (ex, ey) in enumerate(e.tolist())], dim=-1)  # fmt: skip


def momentum_exchange_force(f: torch.Tensor, links: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    r"""F_a = 2 Σ_{xyi} L_{xyi} f_{xyi} e_{ia}  (x, y, i summed): force on the body in lattice units.   → (2,)"""
    return 2 * torch.einsum("xyi,xyi,ia->a", links, f, e)


def viscosity_to_tau(nu: float) -> float:
    """Relaxation time from the lattice viscosity: ν = (τ - ½)/3."""
    return 3 * nu + 0.5


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/fluids/cylinder_drag/tensors.py
"""Build the concrete tensors: the D2Q9 lattice, the cylinder mask, its boundary links and the initial populations."""

import torch

from .equation import equilibrium


def lattice(device: torch.device) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """D2Q9: velocities e_ia (9, 2), weights w_i (9,) and the index of the opposite velocity opp[i] (9,).

    i = 0 is rest, 1-4 the axis directions, 5-8 the diagonals.
    """
    e = torch.tensor(
        [[0, 0], [1, 0], [0, 1], [-1, 0], [0, -1], [1, 1], [-1, 1], [-1, -1], [1, -1]],
        device=device, dtype=torch.float32,
    )  # fmt: skip
    w = torch.tensor([4 / 9] + [1 / 9] * 4 + [1 / 36] * 4, device=device)
    opp = torch.tensor([0, 3, 4, 1, 2, 7, 8, 5, 6], device=device)
    return e, w, opp


def cylinder(nx: int, ny: int, cx: float, cy: float, radius: float, device: torch.device) -> torch.Tensor:
    """Solid mask (Nx, Ny), True inside the cylinder."""
    x = torch.arange(nx, device=device)[:, None]
    y = torch.arange(ny, device=device)[None, :]
    return (x - cx) ** 2 + (y - cy) ** 2 <= radius**2


def boundary_links(solid: torch.Tensor, e: torch.Tensor) -> torch.Tensor:
    """links[x, y, i] = 1 where a population moving along e_i just streamed from a fluid node into solid node (x, y).

    These are the links that carry momentum into the body: (x, y) is solid and (x, y) - e_i is fluid.
    """
    links = []
    for ex, ey in e.int().tolist():
        came_from_fluid = ~torch.roll(solid, shifts=(ex, ey), dims=(0, 1))  # fluid at (x, y) - e_i
        links.append(solid & came_from_fluid)
    return torch.stack(links, dim=-1).float()  # (Nx, Ny, 9)


def initial_populations(
    nx: int, ny: int, u_in: float, e: torch.Tensor, w: torch.Tensor, device: torch.device
) -> torch.Tensor:
    """f_xyi at equilibrium with uniform flow u_in, plus a small sideways kick that triggers the shedding sooner."""
    rho = torch.ones(nx, ny, device=device)
    u = torch.zeros(nx, ny, 2, device=device)
    u[..., 0] = u_in
    y = torch.arange(ny, device=device)
    u[..., 1] = (
        0.2 * u_in * torch.cos(2 * torch.pi * y / ny)[None, :]
    )  # even in y about the centre: breaks the mirror symmetry
    return equilibrium(rho, u, e, w)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/fluids/cylinder_drag/solve.py
"""The lattice Boltzmann loop: collide → stream → measure force → bounce back → inlet/outlet."""

import torch

from problems._common.progress import steps

from .equation import collide, density, equilibrium, momentum, momentum_exchange_force, stream


def macroscopic(f: torch.Tensor, e: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """ρ (Nx, Ny) and u (Nx, Ny, 2) from the populations."""
    rho = density(f)
    return rho, momentum(f, e) / rho[..., None]


def vorticity(u: torch.Tensor) -> torch.Tensor:
    """ω = ∂_x u_y - ∂_y u_x by central differences: (Nx, Ny, 2) → (Nx, Ny)."""
    return torch.gradient(u[..., 1], dim=0)[0] - torch.gradient(u[..., 0], dim=1)[0]


def simulate(
    f: torch.Tensor, solid: torch.Tensor, links: torch.Tensor, e: torch.Tensor, w: torch.Tensor, opp: torch.Tensor,
    tau: float, u_in: float, n_steps: int, save_every: int, open_ends: bool = True,
) -> dict:  # fmt: skip
    """Run n_steps lattice updates. Returns force history (n_steps, 2), vorticity snapshots and final f.

    open_ends=True: equilibrium inflow at x = 0 and a copy-the-neighbour outflow at x = Nx-1.
    open_ends=False: fully periodic box (used by the tests to check exact mass conservation).
    """
    fluid = ~solid[..., None]  # (Nx, Ny, 1)
    ny = f.shape[1]
    inflow = equilibrium(
        torch.ones(1, ny, device=f.device), torch.tensor([u_in, 0.0], device=f.device).expand(1, ny, 2), e, w
    )
    forces, snapshots, times = [], [], []
    for n in steps(n_steps, "lattice steps"):
        rho, u = macroscopic(f, e)
        f = torch.where(fluid, collide(f, equilibrium(rho, u, e, w), tau), f)  # no collision inside the solid
        f = stream(f, e)
        forces.append(momentum_exchange_force(f, links, e))  # populations that just hit the cylinder
        f = torch.where(solid[..., None], f[..., opp], f)  # bounce-back: reverse every population in the solid
        if open_ends:
            f[0] = inflow[0]
            f[-1] = f[-2]
        if n % save_every == 0:
            times.append(n)
            snapshots.append(vorticity(u).masked_fill(solid, float("nan")))
    return {"force": torch.stack(forces), "vorticity": torch.stack(snapshots), "steps": torch.tensor(times), "f": f}


def coefficients(force: torch.Tensor, u_in: float, diameter: float) -> torch.Tensor:
    """C_D, C_L = F / (½ ρ U² D) with ρ = 1: (T, 2) → (T, 2)."""
    return force / (0.5 * u_in**2 * diameter)


def strouhal(lift: torch.Tensor, u_in: float, diameter: float) -> float:
    """St = f D / U from the dominant frequency of the lift coefficient over the second half of the run.

    The signal is windowed (Hann) and zero-padded 16× so the spectrum peak is located finely.
    """
    signal = lift[len(lift) // 2 :]
    signal = (signal - signal.mean()) * torch.hann_window(len(signal), device=signal.device)
    n_fft = 16 * len(signal)
    spectrum = torch.fft.rfft(signal, n=n_fft).abs()
    spectrum[0] = 0
    frequency = int(spectrum.argmax()) / n_fft  # cycles per lattice step
    return frequency * diameter / u_in


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/fluids/cylinder_drag/plot.py
"""Top: vorticity of the wake behind the cylinder. Bottom: drag and lift coefficients over time."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Circle

from problems._common.plotting import frame_indices, save_gif


def animate(
    snapshots: torch.Tensor, snapshot_steps: torch.Tensor, coeffs: torch.Tensor, centre: tuple[float, float],
    radius: float, u_in: float, path: Path,
) -> Path:  # fmt: skip
    w = snapshots.cpu().numpy()  # (F, Nx, Ny), NaN inside the cylinder
    steps = snapshot_steps.cpu().numpy()
    c = coeffs.cpu().numpy()  # (T, 2)
    nx, ny = w.shape[1:]
    limit = 2.5 * u_in / (2 * radius)  # vorticity scale U/D
    late = c[len(c) // 2 :]

    fig, (ax, ax2) = plt.subplots(2, 1, figsize=(7.5, 4.4), gridspec_kw={"height_ratios": [1.3, 1]})
    cmap = plt.get_cmap("RdBu_r", 15).with_extremes(bad="#9ca3af")  # grey inside the cylinder (NaN)
    image = ax.imshow(w[0].T, origin="lower", cmap=cmap, vmin=-limit, vmax=limit, extent=(0, nx, 0, ny),
                      interpolation="nearest")  # fmt: skip
    ax.add_patch(Circle(centre, radius, fc="#9ca3af", ec="#374151", lw=1.2))
    ax.set_xticks([])
    ax.set_yticks([])
    ax.grid(False)
    title = ax.set_title(" ", fontsize=10)

    ax2.set_xlim(0, len(c))
    ax2.set_ylim(min(-1.2, 1.5 * late[:, 1].min()), max(3.0, 1.3 * late[:, 0].max()))
    ax2.axhline(0, color="#9ca3af", lw=0.8)
    ax2.set_xlabel("lattice time step")
    drag = ax2.plot([], [], color="#dc2626", lw=1.4, label="drag C_D")[0]
    lift = ax2.plot([], [], color="#2563eb", lw=1.4, label="lift C_L")[0]
    ax2.legend(loc="upper right", fontsize=8, ncol=2)
    t = np.arange(len(c))

    def update(k: int):
        image.set_data(w[k].T)
        n = int(steps[k])
        drag.set_data(t[: n + 1], c[: n + 1, 0])
        lift.set_data(t[: n + 1], c[: n + 1, 1])
        title.set_text(f"vorticity   step {n}   C_D = {c[max(0, n - 200) : n + 1, 0].mean():.2f}")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(steps), 90), path, fps=15)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    're': 100.0,
    'steps': 5000
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.fluids.cylinder_drag")]:
    del sys.modules[name]
main = importlib.import_module("problems.fluids.cylinder_drag.main")
problem_dir = Path("problems/fluids/cylinder_drag")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
